In [21]:
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from explain.eval.score.syntax_score import SyntaxEvaluator
from explain.eval.score.accuracy_score import AccuracyEvaluator
from explain.eval.score.structure_explain import StructureExplain
from explain.eval.score.structural_score import StructuralEvaluator
from transformers import logging
logging.set_verbosity_error() 

# Load data and evaluator

In [ ]:
os.getcwd()
# Set the path to your hooke-explain directory
os.chdir('/mnt/ps/home/CORP/XX/hooke-explain/')
os.getcwd()

'/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain'

In [14]:
# load data
with open(os.path.join('output/structure_explain/vanilla' , "vanilla_[].json"), "r") as f:
    gen_data = json.load(f)

gt_data = pd.read_csv('data/curation_v1/results/structure-explain-results-v3-claude4.csv')
gt_data = gt_data.to_dict(orient='records')

gt_data = gt_data[:10]
gen_data = gen_data[:10]
# structure explain object
structure_explain = StructureExplain(gen_data)
gt_structure_explain = StructureExplain(gt_data)
syntax_evaluator = SyntaxEvaluator()
accuracy_evaluator = AccuracyEvaluator()
structural_evaluator = StructuralEvaluator()

# Compute metrics

In [ ]:
scores = []

for i, (gt, gen) in enumerate(tqdm(zip(gt_structure_explain, structure_explain), total=len(gen_data))):
    cur_score_dict = {'index': i}
    score = syntax_evaluator.primitive_validity(gen['explain']) 
    cur_score_dict['primitive_validity'] = score
    # token accuracy for structure hypothesis
    score = accuracy_evaluator.token_based_accuracy(gt['explain'], gen['explain'], 'structure_hypothesis')
    cur_score_dict['token_accuracy/structure_hypothesis'] = score
    # token accuracy for paragraph
    score = accuracy_evaluator.token_based_accuracy(gt['answer'], gen['answer'], 'paragraph')
    cur_score_dict['token_accuracy/paragraph'] = score
    # schema validity
    score = syntax_evaluator.schema_validity(gen['raw_response'])
    cur_score_dict['schema_validity'] = score
    scores.append(cur_score_dict)

    # id coherence
    score = syntax_evaluator.id_coherence(gen['explain'], gen['dag'])
    cur_score_dict['id_coherence'] = score
    # dag well formed
    score = syntax_evaluator.dag_well_formed(gen['dag'])
    cur_score_dict['dag_well_formed'] = score


    # edge type accuracy
    score = structural_evaluator.edge_type_accuracy(gt['dag'], gen['dag'])
    cur_score_dict['edge_type_accuracy'] = score
    # primitive level f1
    score = accuracy_evaluator.primitive_level_f1(gt['explain'], gen['explain'])
    cur_score_dict['primitive_level_f1'] = score
    # argument similarity for paragraph
    score = accuracy_evaluator.argument_similarity(gt['answer'], gen['answer'])
    cur_score_dict['argument_similarity/paragraph'] = score
    # argument similarity for structure hypothesis
    score = accuracy_evaluator.argument_similarity(gt['explain'], gen['explain'])
    cur_score_dict['argument_similarity/structure_hypothesis'] = score

  0%|          | 0/10 [00:00<?, ?it/s]/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 10/10 [00:25<00:00,  2.51s/it]


# Show metrics

In [23]:
for key in scores[0].keys():
    if key == 'index':
        continue
    if 'token_accuracy' in key:
        for k in scores[0][key].keys():
            print(key, k, round(np.mean([score[key][k] for score in scores]), 4))
    else:
        print(key, round(np.mean([score[key] for score in scores]), 4))

syntax 0.395
token_accuracy/structure_hypothesis bleu_micro 0.4621
token_accuracy/structure_hypothesis rouge1_micro 0.5825
token_accuracy/structure_hypothesis rouge2_micro 0.3058
token_accuracy/structure_hypothesis rougeL_micro 0.4328
token_accuracy/structure_hypothesis meteor_micro 0.1679
token_accuracy/structure_hypothesis bleu_macro 0.4572
token_accuracy/structure_hypothesis rouge1_macro 0.4683
token_accuracy/structure_hypothesis rouge2_macro 0.2653
token_accuracy/structure_hypothesis rougeL_macro 0.4423
token_accuracy/structure_hypothesis meteor_macro 0.1082
token_accuracy/paragraph bleu_micro 0.0297
token_accuracy/paragraph rouge1_micro 0.4447
token_accuracy/paragraph rouge2_micro 0.1041
token_accuracy/paragraph rougeL_micro 0.2477
token_accuracy/paragraph meteor_micro 0.2496
token_accuracy/paragraph bleu_macro 0.0341
token_accuracy/paragraph rouge1_macro 0.1946
token_accuracy/paragraph rouge2_macro 0.0434
token_accuracy/paragraph rougeL_macro 0.1558
token_accuracy/paragraph meteo